In [ ]:
# GO enrichment (gseapy prerank) on Mac4 markers; thesis figure 4.4.14.
import scanpy as sc
import numpy as np
import pandas as pd  
import matplotlib.pyplot as plt
import matplotlib
import anndata as ad
import scvelo as scv
import palantir as pl

In [ ]:
# Load data
adata = sc.read('/home/yanjun2/scanpy/8_figures/inte_palantir.h5ad')

In [ ]:
sc.settings.figdir = "/home/yanjun2/scanpy/8_figures"
sc.set_figure_params(scanpy=True, dpi=300, dpi_save=300, 
                         frameon=True, vector_friendly=True, fontsize=10, figsize= (5,5), color_map=None, format='pdf', 
                         facecolor=None, transparent=False, ipython_format='png2x')

In [ ]:
# Differential expression
sc.tl.rank_genes_groups(adata, groupby="leiden_res_0_8", method="wilcoxon", layer = 'data')
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {f"{group}_{key}": result[key][group]
    for group in groups for key in ['names', 'scores','pvals','pvals_adj', 'logfoldchanges']}
)

In [ ]:
df.to_csv('/home/yanjun2/scanpy/8_figures/mac4markers.csv')

In [ ]:
df.iloc[:, 15:20]

In [ ]:
import gseapy as gp

In [ ]:
df_rank = df.loc[:, ['Mac4_names', 'Mac4_scores']]

In [ ]:
gsea_res = gp.prerank(rnk = df_rank, gene_sets = 'GO_Biological_Process_2021', seed = 1024)

In [ ]:
out_list = []

for term in gsea_res.results:
    p = gsea_res.results[term]['pval']
    fdr = gsea_res.results[term]['fdr']
    nes = gsea_res.results[term]['nes']
    gene = gsea_res.results[term]['lead_genes']
    out_list.append([term, p, fdr, nes, gene])

df_out = pd.DataFrame(out_list, columns = ['Term', 'P_value', 'FDR', 'NES', 'Gene']).sort_values('NES', ascending=False).reset_index(drop=True)

In [ ]:
df_out

In [ ]:
df_out.to_csv('/home/yanjun2/scanpy/8_figures/mac4bp.csv')